### Рекомендации статей Т‑Ж: i2i по **текстовым эмбеддингам** (retrieval) + rerank/mix

Этот ноутбук — основной артефакт пайплайна i2i, где retrieval делается по **готовым эмбеддингам текста** статьи (файл `user_articles_embeddings.csv`, 1024d).

- **Retrieval (similar candidates)**: top‑N соседей по cosine в эмбеддингах текста.
- **Explore candidates**: отдельный широкий пул «популярных/качественных» статей (не по похожести).
- **Сборка карусели (K=12)**: 9 `similar` + 2 `explore` (тот же департамент, **другая рубрика**) + 1 `explore` (другой департамент, с блэклистом переходов).

Важно: текущая версия — **MVP**. Сильные улучшения: кэш/FAISS для retrieval, LTR rerank по логам показов/кликов/quality.


### Зависимости

Если запускаете в среде без библиотек — установите пакеты.


In [ ]:
# При необходимости раскомментируйте:
# %pip install -U pandas numpy scikit-learn pyarrow lightgbm


### Загрузка `tj_article.csv` и подготовка признаков

- CSV разделён `;`
- Числа иногда с запятой как десятичным разделителем


In [ ]:
from __future__ import annotations

import math
import os
import re
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.neighbors import NearestNeighbors


def safe_str(x: object) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def load_articles(path: str = "tj_article.csv") -> pd.DataFrame:
    read_kwargs = dict(sep=";", low_memory=False)
    for enc in ("utf-8", "utf-8-sig", "cp1251"):
        try:
            df = pd.read_csv(path, encoding=enc, **read_kwargs)
            break
        except UnicodeDecodeError:
            df = None
    if df is None:
        df = pd.read_csv(path, encoding="utf-8", encoding_errors="replace", **read_kwargs)

    df["article_id"] = df["article_id"].map(safe_str)
    for col in [
        "article_base__title",
        "article_base__department",
        "article_base__rubric",
        "article_base__author_id",
        "article_base__author_name",
    ]:
        if col in df.columns:
            df[col] = df[col].map(safe_str)

    numeric_prefixes = ("article_stats__", "article_dates__", "article_author__")
    for col in df.columns:
        if not col.startswith(numeric_prefixes):
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            s = df[col].astype(str).str.replace(",", ".", regex=False)
            df[col] = pd.to_numeric(s, errors="coerce")

    return df


df = load_articles("tj_article.csv")
df.shape


### Загрузка эмбеддингов текста (`user_articles_embeddings.csv`)

Файл большой. Рекомендуется один раз конвертировать в быстрый формат (memmap):

- `embeddings_cache/ids.txt`
- `embeddings_cache/embeddings.f32`
- `embeddings_cache/shape.txt`

После этого загрузка и retrieval становятся быстрыми.


In [ ]:
def build_embeddings_memmap(
    csv_path: str = "user_articles_embeddings.csv",
    *,
    out_dir: str = "embeddings_cache",
) -> None:
    import csv

    os.makedirs(out_dir, exist_ok=True)
    ids_path = os.path.join(out_dir, "ids.txt")
    emb_path = os.path.join(out_dir, "embeddings.f32")
    shape_path = os.path.join(out_dir, "shape.txt")

    with open(csv_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        r = csv.reader(f)
        header = next(r)
        dim = len(header) - 1
        n = sum(1 for _ in r)

    X = np.memmap(emb_path, dtype="float32", mode="w+", shape=(n, dim))
    ids: list[str] = []

    with open(csv_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        r = csv.reader(f)
        next(r)
        j = 0
        for row in r:
            if not row:
                continue
            aid = safe_str(row[0])
            if not aid:
                continue
            v = np.asarray(row[1 : 1 + dim], dtype="float32")
            X[j] = v
            ids.append(aid)
            j += 1

    X.flush()
    with open(ids_path, "w", encoding="utf-8") as f:
        for aid in ids:
            f.write(aid + "
")
    with open(shape_path, "w", encoding="utf-8") as f:
        f.write(f"{j},{dim}
")

    print("Wrote", j, "rows to", out_dir)


def load_embeddings_memmap(out_dir: str = "embeddings_cache") -> tuple[list[str], np.ndarray]:
    ids_path = os.path.join(out_dir, "ids.txt")
    emb_path = os.path.join(out_dir, "embeddings.f32")
    shape_path = os.path.join(out_dir, "shape.txt")

    n_str, dim_str = open(shape_path, "r", encoding="utf-8").read().strip().split(",")
    n, dim = int(n_str), int(dim_str)

    ids = [line.strip() for line in open(ids_path, "r", encoding="utf-8") if line.strip()]
    X = np.memmap(emb_path, dtype="float32", mode="r", shape=(n, dim))
    X = np.asarray(X)

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    X = (X / norms).astype(np.float32)
    return ids, X


# Пример (не запускаем автоматически):
# build_embeddings_memmap('user_articles_embeddings.csv', out_dir='embeddings_cache')
# X_ids, X = load_embeddings_memmap('embeddings_cache')


### Align эмбеддингов с таблицей статей + retrieval индекс


In [ ]:
@dataclass(frozen=True)
class Artifacts:
    df: pd.DataFrame
    article_id_to_row: dict[str, int]
    nn: NearestNeighbors
    X: np.ndarray  # normalized text embeddings


def fit_retriever_from_text_embeddings(df: pd.DataFrame, X_ids: list[str], X: np.ndarray) -> Artifacts:
    id_to_emb_row = {aid: i for i, aid in enumerate(X_ids) if aid}

    keep_rows: list[int] = []
    emb_rows: list[int] = []
    for i, aid in enumerate(df["article_id"].map(safe_str).tolist()):
        j = id_to_emb_row.get(aid)
        if j is None:
            continue
        keep_rows.append(i)
        emb_rows.append(j)

    df2 = df.iloc[keep_rows].copy().reset_index(drop=True)
    X2 = X[np.asarray(emb_rows, dtype=np.int64)]

    nn = NearestNeighbors(metric="cosine", algorithm="brute")
    nn.fit(X2)

    article_id_to_row = {aid: i for i, aid in enumerate(df2["article_id"].tolist()) if aid}
    return Artifacts(df=df2, article_id_to_row=article_id_to_row, nn=nn, X=X2)


# Пример (не запускаем автоматически):
# X_ids, X = load_embeddings_memmap('embeddings_cache')
# art = fit_retriever_from_text_embeddings(df, X_ids, X)
# df2 = art.df


### Anti-duplicate/series: `title_norm`


In [ ]:
_RE_YEAR = re.compile(r"\b(19|20)\d{2}\b")
_RE_NUM = re.compile(r"\b\d+([.,]\d+)?\b")
_RE_SPACE = re.compile(r"\s+")


def normalize_title_for_dedup(title: str) -> str:
    t = safe_str(title).lower().replace("ё", "е")
    t = _RE_YEAR.sub(" <year> ", t)
    t = _RE_NUM.sub(" <num> ", t)
    t = re.sub(r"["'“”«»()\[\]{}:;,.!?/\\|—–-]+", " ", t)
    t = _RE_SPACE.sub(" ", t).strip()
    return t


### Priors качества/тренда/свежести (из агрегатов)


In [ ]:
def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))


def quality_prior(row: pd.Series) -> float:
    like_rate = row.get("article_stats__like_rate", np.nan)
    comment_rate = row.get("article_stats__comment_rate", np.nan)
    favs = row.get("article_stats__stats_favorites", np.nan)
    views = row.get("article_stats__stats_views", np.nan)

    lr = float(like_rate) if pd.notna(like_rate) else 0.0
    cr = float(comment_rate) if pd.notna(comment_rate) else 0.0
    fav_rate = (float(favs) / max(float(views), 1.0)) if pd.notna(favs) and pd.notna(views) else 0.0

    return float(_sigmoid(12.0 * (0.7 * lr + 0.3 * fav_rate) + 4.0 * cr))


def trending_prior(row: pd.Series) -> float:
    views = row.get("article_stats__stats_views", np.nan)
    days = row.get("article_dates__days_since_published", np.nan)
    v = float(views) if pd.notna(views) else 0.0
    d = float(days) if pd.notna(days) else 365.0
    return float(math.log1p(v) / math.sqrt(d + 1.0))


def freshness(row: pd.Series) -> float:
    days = row.get("article_dates__days_since_published", np.nan)
    d = float(days) if pd.notna(days) else 365.0
    return float(1.0 / (1.0 + d / 30.0))


### Двухэтапная генерация кандидатов: `similar` и `explore`

Дополнительно для **similar** можно подключить поведенческий retrieval по **coview**:

- берём top‑M статей, которые пользователи часто читают после источника (не из клика по рекомендациям);
- объединяем с кандидатами из текстовых эмбеддингов;
- дальше работает тот же rerank/LTR и продуктовые правила.


In [ ]:
DEPT_CROSS_BLACKLIST: dict[str, set[str]] = {
    "Медицина": {"Еда"},
}


def build_explore_pools(df: pd.DataFrame, *, top_m: int = 3000) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    tmp = df.copy()
    tmp["quality"] = tmp.apply(quality_prior, axis=1)
    tmp["trend"] = tmp.apply(trending_prior, axis=1)
    tmp["explore_score"] = 0.6 * tmp["quality"] + 0.4 * tmp["trend"]

    per_dept: dict[str, pd.DataFrame] = {}
    if "article_base__department" in tmp.columns:
        for dept, g in tmp.groupby("article_base__department", dropna=False):
            d = safe_str(dept)
            if not d:
                continue
            per_dept[d] = g.sort_values("explore_score", ascending=False).head(top_m)

    global_pool = tmp.sort_values("explore_score", ascending=False).head(top_m)
    return per_dept, global_pool


def save_coview_index(index: dict[str, list[tuple[str, float]]], path: str) -> None:
    """Сохраняет coview индекс на диск (gzip+pickle)."""
    import gzip
    import pickle

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with gzip.open(path, "wb") as f:
        pickle.dump(index, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_coview_index(path: str) -> dict[str, list[tuple[str, float]]]:
    """Загружает coview индекс с диска (gzip+pickle)."""
    import gzip
    import pickle

    with gzip.open(path, "rb") as f:
        return pickle.load(f)


def build_coview_index(
    parquet_path: str,
    *,
    top_k_per_source: int = 200,
    min_count: int = 2,
    source_type_value: str = "coview",
    max_rows: int | None = None,
    batch_size: int = 200_000,
    score_mode: str = "p_b_given_a",
    alpha: float = 1.0,
) -> dict[str, list[tuple[str, float]]]:
    """Строит i2i индекс по coview логам: source -> [(target, score), ...].

    Почему не просто count:
    - сырые счётчики сильно зависят от популярности источника;
    - более стабильный дефолт — условная вероятность p(b|a) со сглаживанием.

    score_mode:
    - "p_b_given_a": log(p(b|a)) ~ log(cnt+α) - log(total_src+α·K)
    - "lift": log-lift ~ log(p(b|a)) - log(p(b)) (приближённо; p(b) считаем по целям)

    Реализация потоковая и ограничивает память: для каждого source храним только
    ограниченное число top кандидатов.
    """
    import pyarrow.dataset as ds

    dataset = ds.dataset(parquet_path, format="parquet")
    need = {"source_article_id", "target_article_id", "source_type"}
    missing = need - set(dataset.schema.names)
    if missing:
        raise KeyError(f"coview logs parquet missing cols: {missing}")

    scanner = dataset.scanner(columns=["source_article_id", "target_article_id", "source_type"], batch_size=batch_size)

    # src -> dict[tgt] = cnt
    store: dict[str, dict[str, int]] = {}
    src_total: dict[str, int] = {}
    tgt_total: dict[str, int] = {}
    global_total = 0
    seen = 0

    def _prune(d: dict[str, int]) -> dict[str, int]:
        if len(d) <= top_k_per_source * 3:
            return d
        items = sorted(d.items(), key=lambda x: x[1], reverse=True)[: top_k_per_source * 2]
        return dict(items)

    for batch in scanner.to_batches():
        pdf = batch.to_pandas(strings_to_categorical=False)
        if source_type_value:
            pdf = pdf[pdf["source_type"].astype(str) == source_type_value]
        if len(pdf) == 0:
            continue

        for src, tgt in zip(pdf["source_article_id"].astype(str).tolist(), pdf["target_article_id"].astype(str).tolist()):
            src = safe_str(src)
            tgt = safe_str(tgt)
            if not src or not tgt or src == tgt:
                continue

            global_total += 1
            src_total[src] = src_total.get(src, 0) + 1
            tgt_total[tgt] = tgt_total.get(tgt, 0) + 1

            m = store.get(src)
            if m is None:
                m = {}
                store[src] = m
            m[tgt] = m.get(tgt, 0) + 1
            if len(m) > top_k_per_source * 4:
                store[src] = _prune(m)

        seen += len(pdf)
        if max_rows is not None and seen >= max_rows:
            break

    out: dict[str, list[tuple[str, float]]] = {}

    for src, m in store.items():
        items = [(t, c) for t, c in m.items() if c >= min_count]
        if not items:
            continue

        tot = float(src_total.get(src, 0))
        K = float(len(items))

        scored: list[tuple[str, float]] = []
        for t, c in items:
            # log p(b|a)
            log_p_b_a = float(math.log((c + alpha) / max(tot + alpha * K, 1.0)))
            if score_mode == "lift":
                pb = (tgt_total.get(t, 0) + alpha) / max(global_total + alpha * max(len(tgt_total), 1), 1.0)
                score = log_p_b_a - float(math.log(pb))
            else:
                score = log_p_b_a
            scored.append((t, score))

        scored.sort(key=lambda x: x[1], reverse=True)
        out[src] = scored[:top_k_per_source]

    return out


def get_similar_candidates(
    art: Artifacts,
    source_idx: int,
    *,
    top_n: int = 800,
    coview_index: dict[str, list[tuple[str, int]]] | None = None,
    coview_top_m: int = 200,
    coview_weight: float = 0.35,
) -> tuple[np.ndarray, np.ndarray]:
    """Retrieval кандидатов для similar.

    - базовый источник: cosine-NN по текстовым эмбеддингам
    - доп. источник: coview i2i (если передан индекс)

    Возвращает массивы индексов df и их "similarity-like" score.
    """
    distances, indices = art.nn.kneighbors(
        art.X[source_idx : source_idx + 1],
        n_neighbors=min(top_n + 1, art.df.shape[0]),
    )
    distances = distances.ravel()
    indices = indices.ravel()
    mask = indices != source_idx
    indices = indices[mask]
    sims = (1.0 - distances[mask]).astype(np.float32)

    # merge coview
    if coview_index is not None:
        src_id = safe_str(art.df.iloc[source_idx].get("article_id", ""))
        if src_id and src_id in coview_index:
            items = coview_index[src_id][:coview_top_m]
            if items:
                # items: (target_article_id, score) где score ~ log(p(b|a)) или lift
                # Приведём к [0,1] через сигмоиду и домножим на coview_weight.
                for tgt_id, sc in items:
                    j = art.article_id_to_row.get(safe_str(tgt_id))
                    if j is None or j == source_idx:
                        continue
                    z = float(sc)
                    cov_sim = coview_weight * float(1.0 / (1.0 + math.exp(-z)))
                    # if already in NN list, take max
                    hit = np.where(indices == j)[0]
                    if len(hit):
                        sims[hit[0]] = max(float(sims[hit[0]]), float(cov_sim))
                    else:
                        indices = np.append(indices, np.int64(j))
                        sims = np.append(sims, np.float32(cov_sim))

    return indices.astype(np.int64), sims.astype(np.float32)


def get_explore_candidates(
    art: Artifacts,
    source_idx: int,
    *,
    per_dept_pool: dict[str, pd.DataFrame],
    global_pool: pd.DataFrame,
    pool_size_same: int = 2000,
    pool_size_cross: int = 2000,
) -> pd.DataFrame:
    src = art.df.iloc[source_idx]
    src_dept = safe_str(src.get("article_base__department", ""))
    src_id = safe_str(src.get("article_id", ""))

    same = per_dept_pool.get(src_dept)
    if same is None:
        same = global_pool
    same = same.head(pool_size_same).copy()

    cross = global_pool.head(pool_size_cross).copy()
    if src_dept and "article_base__department" in cross.columns:
        cross = cross[cross["article_base__department"].map(safe_str) != src_dept]

    forbidden = DEPT_CROSS_BLACKLIST.get(src_dept, set())
    if forbidden and "article_base__department" in cross.columns:
        cross = cross[~cross["article_base__department"].map(safe_str).isin(forbidden)]

    out = pd.concat([same, cross], ignore_index=True)
    if src_id and "article_id" in out.columns:
        out = out[out["article_id"].map(safe_str) != src_id]
    return out


### Сборка карусели: `rerank_and_diversify_two_stage` (K=12, explore=3)


In [ ]:
def rerank_and_diversify_two_stage(
    art: Artifacts,
    source_idx: int,
    sim_indices: np.ndarray,
    sim_sims: np.ndarray,
    explore_df: pd.DataFrame,
    *,
    k: int = 20,
    explore_slots: int = 4,
    max_same_author: int = 2,
    max_same_rubric: int = 6,
    avoid_same_title_norm: bool = True,
    w_sim: float = 0.72,
    w_quality: float = 0.18,
    w_trend: float = 0.06,
    w_fresh: float = 0.04,
    ltr_model: object | None = None,
    max_candidate_age_days: float | None = None,
) -> list[dict]:
    """Сборка карусели из двух пулов.

    Если задан `max_candidate_age_days`, отбрасываем кандидатов старше
    (по `article_dates__days_since_published`; NaN не отбрасываем).

    Если передан `ltr_model`, он используется как финальный скор ранжирования
    (вместо эвристического w_sim/w_quality/...). Квоты слотов, safety и diversity
    остаются такими же.

    Ожидаемый интерфейс модели: `predict(X: np.ndarray) -> np.ndarray`.
    """
    df = art.df
    src = df.iloc[source_idx]
    src_id = safe_str(src.get("article_id", ""))
    src_dept = safe_str(src.get("article_base__department", ""))
    src_rubric = safe_str(src.get("article_base__rubric", ""))
    src_title_norm = safe_str(src.get("title_norm", ""))

    c = df.iloc[sim_indices].copy()
    c["sim"] = sim_sims
    c["quality"] = c.apply(quality_prior, axis=1)
    c["trend"] = c.apply(trending_prior, axis=1)
    c["fresh"] = c.apply(freshness, axis=1)

    _age_col = "article_dates__days_since_published"
    if max_candidate_age_days is not None and _age_col in c.columns:
        _ad = pd.to_numeric(c[_age_col], errors="coerce")
        _keep = _ad.isna() | (_ad <= float(max_candidate_age_days))
        _cf = c.loc[_keep].copy()
        if len(_cf) > 0:
            c = _cf

    thr = max(0.08, float(np.quantile(c["sim"].values, 0.5)))
    gated = c[c["sim"] >= thr].copy()
    if len(gated) > 0:
        c = gated

    c["cand_dept"] = c.get("article_base__department", "").map(safe_str)
    if src_dept:
        sd = c[c["cand_dept"] == src_dept].copy()
        od = c[c["cand_dept"] != src_dept].copy()
    else:
        sd = c
        od = c.iloc[0:0]

    def _rank01(s: pd.Series) -> pd.Series:
        if len(s) <= 1:
            return pd.Series([0.5] * len(s), index=s.index)
        return s.rank(pct=True)

    # source context features (must match training)
    _sv = src.get("article_stats__stats_views", np.nan)
    src_views_raw = float(_sv) if pd.notna(_sv) else 0.0
    src_log_views = float(math.log1p(max(src_views_raw, 0.0)))

    _slr = src.get("article_stats__like_rate", np.nan)
    src_like_rate = float(_slr) if pd.notna(_slr) else 0.0

    src_fresh = float(freshness(src))

    _sad = src.get("article_dates__days_since_published", np.nan)
    src_age_days = float(_sad) if pd.notna(_sad) else 365.0

    src_author = safe_str(src.get("article_base__author_id", ""))

    def _ltr_predict(sub: pd.DataFrame) -> np.ndarray:
        # Фичи должны совпадать с обучением в `build_ltr_dataset_from_logs`.
        cand_rub = sub.get("article_base__rubric", "").map(safe_str)
        same_dept = (sub.get("cand_dept", "").map(safe_str) == src_dept).astype(np.float32)
        same_rub = (cand_rub == src_rubric).astype(np.float32)
        pos = np.zeros(len(sub), dtype=np.float32)  # в проде позиция неизвестна до ранжирования

        views_raw = pd.to_numeric(sub.get("article_stats__stats_views", 0.0), errors="coerce").fillna(0.0).astype(np.float32).values
        comm_raw = pd.to_numeric(sub.get("article_stats__stats_comments", 0.0), errors="coerce").fillna(0.0).astype(np.float32).values
        favs_raw = pd.to_numeric(sub.get("article_stats__stats_favorites", 0.0), errors="coerce").fillna(0.0).astype(np.float32).values

        cand_log_views = np.log1p(np.clip(views_raw, 0, None)).astype(np.float32)
        cand_log_comments = np.log1p(np.clip(comm_raw, 0, None)).astype(np.float32)
        cand_like_rate = pd.to_numeric(sub.get("article_stats__like_rate", 0.0), errors="coerce").fillna(0.0).astype(np.float32).values
        cand_comment_rate = pd.to_numeric(sub.get("article_stats__comment_rate", 0.0), errors="coerce").fillna(0.0).astype(np.float32).values
        cand_fav_rate = (favs_raw / np.maximum(views_raw, 1.0)).astype(np.float32)
        cand_fresh = sub.get("fresh")
        if cand_fresh is None:
            cand_fresh = sub.apply(freshness, axis=1).astype(np.float32).values
        else:
            cand_fresh = pd.to_numeric(cand_fresh, errors="coerce").fillna(0.0).astype(np.float32).values

        cand_age_days = pd.to_numeric(sub.get("article_dates__days_since_published", 365.0), errors="coerce").fillna(365.0).astype(np.float32).values
        cand_author = sub.get("article_base__author_id", "").map(safe_str).astype(str).values
        same_author = (cand_author == src_author).astype(np.float32)
        abs_age_diff_days = np.abs(cand_age_days - np.float32(src_age_days)).astype(np.float32)

        Xf = np.column_stack([
            sub["sim"].astype(np.float32).values,
            cand_log_views,
            cand_like_rate,
            cand_comment_rate,
            cand_log_comments,
            cand_fav_rate,
            cand_fresh,
            same_author,
            abs_age_diff_days,
            same_dept.values,
            same_rub.values,
            np.full(len(sub), float(src_log_views), dtype=np.float32),
            np.full(len(sub), float(src_like_rate), dtype=np.float32),
            np.full(len(sub), float(src_fresh), dtype=np.float32),
            pos,
        ]).astype(np.float32)
        return np.asarray(ltr_model.predict(Xf), dtype=np.float32)

    def _score(sub: pd.DataFrame) -> pd.DataFrame:
        sub = sub.copy()
        if ltr_model is None:
            sub["qual_d"] = _rank01(sub["quality"])
            sub["trend_d"] = _rank01(sub["trend"])
            sub["score"] = w_sim * sub["sim"] + w_quality * sub["qual_d"] + w_trend * sub["trend_d"] + w_fresh * sub["fresh"]
        else:
            sub["score"] = _ltr_predict(sub)

        if avoid_same_title_norm and src_title_norm:
            sub.loc[sub.get("title_norm", "").astype(str) == src_title_norm, "score"] *= 0.20
        return sub

    sd = _score(sd).sort_values("score", ascending=False)
    if len(od):
        od = _score(od)
        if ltr_model is None:
            od["score"] = w_sim * od["sim"] * 0.35
        else:
            od["score"] = od["score"] * 0.85  # мягко предпочитаем same-dept, если модель не уверена
        od = od.sort_values("score", ascending=False)

    c_ordered = pd.concat([sd, od], ignore_index=True)

    explore_slots = min(max(0, explore_slots), k)
    target_sim = max(0, k - explore_slots)

    picked: list[dict] = []
    used_title_norm: set[str] = {src_title_norm} if src_title_norm else set()
    author_cnt: dict[str, int] = {}
    rubric_cnt: dict[str, int] = {}

    def _can_pick_sim(r: pd.Series) -> bool:
        aid = safe_str(r.get("article_base__author_id", ""))
        rub = safe_str(r.get("article_base__rubric", ""))
        tnorm = safe_str(r.get("title_norm", ""))
        if avoid_same_title_norm and tnorm and tnorm in used_title_norm:
            return False
        if aid and author_cnt.get(aid, 0) >= max_same_author:
            return False
        if rub and rubric_cnt.get(rub, 0) >= max_same_rubric:
            return False
        return True

    # Core similar: guarantee at least 1 globally top-similar item (baseline-style),
    # even if diversity caps would block it later.
    core_similar = 1
    if core_similar > 0 and target_sim > 0:
        core_pool = c_ordered.sort_values("sim", ascending=False)
        for _, r in core_pool.iterrows():
            if len(picked) >= min(core_similar, target_sim):
                break
            cid = safe_str(r.get("article_id", ""))
            if not cid:
                continue
            tnorm = safe_str(r.get("title_norm", ""))
            if avoid_same_title_norm and tnorm and tnorm in used_title_norm:
                continue
            picked.append({
                "source_article_id": src_id,
                "candidate_article_id": cid,
                "mix": "similar",
                "score": float(r.get("score", 0.0)),
                "similarity": float(r.get("sim", 0.0)),
                "candidate_title": safe_str(r.get("article_base__title", "")),
                "candidate_department": safe_str(r.get("article_base__department", "")),
                "candidate_rubric": safe_str(r.get("article_base__rubric", "")),
            })
            if tnorm:
                used_title_norm.add(tnorm)
            aid = safe_str(r.get("article_base__author_id", ""))
            if aid:
                author_cnt[aid] = author_cnt.get(aid, 0) + 1
            rub = safe_str(r.get("article_base__rubric", ""))
            if rub:
                rubric_cnt[rub] = rubric_cnt.get(rub, 0) + 1

    picked_ids = {p["candidate_article_id"] for p in picked}

    for _, r in c_ordered.iterrows():
        if len(picked) >= target_sim:
            break
        cid = safe_str(r.get("article_id", ""))
        if not cid or cid in picked_ids:
            continue
        if not _can_pick_sim(r):
            continue
        picked.append({
            "source_article_id": src_id,
            "candidate_article_id": cid,
            "mix": "similar",
            "score": float(r.get("score", 0.0)),
            "similarity": float(r.get("sim", 0.0)),
            "candidate_title": safe_str(r.get("article_base__title", "")),
            "candidate_department": safe_str(r.get("article_base__department", "")),
            "candidate_rubric": safe_str(r.get("article_base__rubric", "")),
        })
        picked_ids.add(cid)
        tnorm = safe_str(r.get("title_norm", ""))
        if tnorm:
            used_title_norm.add(tnorm)
        aid = safe_str(r.get("article_base__author_id", ""))
        if aid:
            author_cnt[aid] = author_cnt.get(aid, 0) + 1
        rub = safe_str(r.get("article_base__rubric", ""))
        if rub:
            rubric_cnt[rub] = rubric_cnt.get(rub, 0) + 1

    picked_ids = {p["candidate_article_id"] for p in picked}

    # similar embeddings for anti-sim
    sim_rows = [art.article_id_to_row[pid] for pid in picked_ids if pid in art.article_id_to_row]
    sim_vecs = art.X[sim_rows] if sim_rows else None

    explore_same_target = max(0, explore_slots - 1) if src_dept else explore_slots
    explore_cross_target = 1 if src_dept and explore_slots > 0 else 0
    explore_same_added = 0
    explore_cross_added = 0

    explore_title_norms: set[str] = set()

    def _max_sim_to_picked_sim(v: np.ndarray) -> float:
        if sim_vecs is None or len(sim_vecs) == 0:
            return 0.0
        return float(np.max(sim_vecs @ v))

    exp = explore_df.copy()
    if max_candidate_age_days is not None and _age_col in exp.columns:
        _ad_e = pd.to_numeric(exp[_age_col], errors="coerce")
        _keep_e = _ad_e.isna() | (_ad_e <= float(max_candidate_age_days))
        _ef = exp.loc[_keep_e].copy()
        if len(_ef) > 0:
            exp = _ef
    if "title_norm" not in exp.columns and "article_base__title" in exp.columns:
        exp["title_norm"] = exp["article_base__title"].map(normalize_title_for_dedup)
    if "explore_score" not in exp.columns:
        exp["quality"] = exp.apply(quality_prior, axis=1)
        exp["trend"] = exp.apply(trending_prior, axis=1)
        exp["explore_score"] = 0.6 * exp["quality"] + 0.4 * exp["trend"]

    if ltr_model is None:
        exp = exp.sort_values("explore_score", ascending=False)
    else:
        # Для LTR ранжируем explore-кандидатов моделью (а не эвристическим explore_score)
        exp = exp.copy()
        exp["cand_dept"] = exp.get("article_base__department", "").map(safe_str)
        exp["fresh"] = exp.apply(freshness, axis=1)

        # similarity к источнику (если есть эмбеддинги)
        src_v = art.X[source_idx]
        cids = exp.get("article_id", "").map(safe_str).tolist()
        sims = np.zeros(len(cids), dtype=np.float32)
        rows = []
        idx_map = {}
        for i, cid in enumerate(cids):
            j = art.article_id_to_row.get(cid)
            if j is None:
                continue
            idx_map[i] = j
            rows.append(j)
        if rows:
            vv = art.X[np.asarray(rows, dtype=np.int64)]
            sims_idx = np.asarray(list(idx_map.keys()), dtype=np.int64)
            sims[sims_idx] = (vv @ src_v).astype(np.float32)
        exp["sim"] = sims

        # LTR score
        exp["score"] = _ltr_predict(exp)
        exp = exp.sort_values("score", ascending=False)

    # Explore sampling to avoid the same global top items for every source.
    # We sample from top-M by explore_score with a deterministic seed from source id.
    explore_sample_pool = 300
    explore_temperature = 0.35

    import hashlib
    seed = int.from_bytes(hashlib.blake2b(src_id.encode("utf-8"), digest_size=8).digest(), "little")
    rng = np.random.default_rng(seed)

    exp = exp.head(int(explore_sample_pool)).copy().reset_index(drop=True)

    def _try_take_explore(r: pd.Series) -> bool:
        nonlocal explore_same_added, explore_cross_added

        if len([x for x in picked if x["mix"] == "explore"]) >= explore_slots:
            return False

        cid = safe_str(r.get("article_id", ""))
        if not cid or cid in picked_ids:
            return False

        tnorm = safe_str(r.get("title_norm", ""))
        if avoid_same_title_norm:
            if src_title_norm and tnorm == src_title_norm:
                return False
            if tnorm and tnorm in explore_title_norms:
                return False

        cand_dept = safe_str(r.get("article_base__department", ""))
        cand_rub = safe_str(r.get("article_base__rubric", ""))

        if src_dept:
            if cand_dept == src_dept:
                if explore_same_added >= explore_same_target:
                    return False
                if src_rubric and cand_rub and cand_rub == src_rubric:
                    return False
                max_sim_thr = 0.55
            else:
                if explore_cross_added >= explore_cross_target:
                    return False
                forbidden = DEPT_CROSS_BLACKLIST.get(src_dept, set())
                if cand_dept and cand_dept in forbidden:
                    return False
                max_sim_thr = 0.45
        else:
            max_sim_thr = 0.55

        vrow = art.article_id_to_row.get(cid)
        if vrow is not None:
            v = art.X[vrow]
            if _max_sim_to_picked_sim(v) > max_sim_thr:
                return False

        # commit
        if src_dept and cand_dept == src_dept:
            explore_same_added += 1
        elif src_dept and cand_dept != src_dept:
            explore_cross_added += 1

        picked.append({
            "source_article_id": src_id,
            "candidate_article_id": cid,
            "mix": "explore",
            "score": float(r.get("explore_score", 0.0)),
            "similarity": float(0.0 if vrow is None else float(np.dot(art.X[source_idx], art.X[vrow]))),
            "candidate_title": safe_str(r.get("article_base__title", "")),
            "candidate_department": cand_dept,
            "candidate_rubric": cand_rub,
        })
        picked_ids.add(cid)
        if tnorm:
            explore_title_norms.add(tnorm)
        return True

    def _sample_idx(df_pool: pd.DataFrame) -> int:
        w = pd.to_numeric(df_pool.get("explore_score", 0.0), errors="coerce").fillna(0.0).astype(float).values
        z = (w - float(np.max(w))) / float(explore_temperature)
        p = np.exp(np.clip(z, -30, 30))
        p = p / max(float(p.sum()), 1e-12)
        return int(rng.choice(len(df_pool), p=p))

    pool = exp.copy().reset_index(drop=True)
    attempts = 0
    while len([x for x in picked if x["mix"] == "explore"]) < explore_slots and len(pool) > 0 and attempts < explore_slots * 50:
        attempts += 1
        idx = _sample_idx(pool)
        r = pool.iloc[idx]
        pool = pool.drop(pool.index[idx]).reset_index(drop=True)
        _try_take_explore(r)

    # Fallback: deterministic scan if sampling couldn't fill all slots
    if len([x for x in picked if x["mix"] == "explore"]) < explore_slots:
        for _, r in exp.iterrows():
            if len([x for x in picked if x["mix"] == "explore"]) >= explore_slots:
                break
            _try_take_explore(r)

    # Final fallback: relax explore quotas/constraints to always fill explore slots.
    if len([x for x in picked if x["mix"] == "explore"]) < explore_slots:
        saved_same, saved_cross = explore_same_target, explore_cross_target
        explore_same_target = explore_slots
        explore_cross_target = explore_slots
        for _, r in exp.iterrows():
            if len([x for x in picked if x["mix"] == "explore"]) >= explore_slots:
                break
            _try_take_explore(r)
        explore_same_target, explore_cross_target = saved_same, saved_cross

    if len([x for x in picked if x["mix"] == "explore"]) < explore_slots and src_dept and src_rubric:
        # Allow same-rubric within dept as a last resort.
        for _, r in exp.iterrows():
            if len([x for x in picked if x["mix"] == "explore"]) >= explore_slots:
                break
            cand_dept = safe_str(r.get("article_base__department", ""))
            cand_rub = safe_str(r.get("article_base__rubric", ""))
            if cand_dept == src_dept and cand_rub == src_rubric:
                rr = r.copy()
                rr["article_base__rubric"] = ""
                _try_take_explore(rr)

    # Interleave explore into the carousel instead of placing it at the end.
    # Default: 1 explore per ~5 positions; tune by changing `explore_gap` below.
    explore_gap = 5

    similar_items = [x for x in picked if x.get("mix") == "similar"]
    explore_items = [x for x in picked if x.get("mix") == "explore"]

    gap = max(2, int(explore_gap))
    if explore_slots > 0:
        raw_pos = [min(k, gap * i) for i in range(1, explore_slots)]
        raw_pos.append(k)
        explore_positions = sorted({p for p in raw_pos if 1 <= p <= k})
    else:
        explore_positions = []

    merged: list[dict] = []
    si = 0
    ei = 0
    for pos in range(1, k + 1):
        want_explore = pos in explore_positions and ei < len(explore_items)
        if want_explore:
            merged.append(explore_items[ei])
            ei += 1
        elif si < len(similar_items):
            merged.append(similar_items[si])
            si += 1
        elif ei < len(explore_items):
            merged.append(explore_items[ei])
            ei += 1
        else:
            break

    # Add explicit carousel rank (position).
    for i, row in enumerate(merged, start=1):
        row["rank"] = i

    return merged[:k]


### LTR rerank: обучение на логах показов/кликов и применение в карусели

Ниже — минимальный продовый контур:

- читаем Parquet с логами (показы кандидатов в карусели);
- строим обучающую выборку по показам (query = один показ карусели под статьёй-источником);
- обучаем `LightGBMRanker` (LambdaRank);
- используем модель как финальный скор ранжирования внутри `rerank_and_diversify_two_stage`.

Важно: лог может содержать несколько разных блоков рекомендаций (`entity_type`) и повторные показы в рамках одной сессии. Поэтому для обучения мы определяем **impression_key** как `(visit_id, source_article_id, hit_dttm, entity_type)`.


In [ ]:
# Оффлайн‑оценка retrieval: Recall@N на сэмпле логов
#
# Сравниваем 3 режима candidate generation для similar:
# - embeddings-only
# - coview-only
# - union (embeddings + coview)

import pyarrow.dataset as ds


def sample_impressions_from_logs(
    parquet_path: str = LOGS_PARQUET_PATH,
    *,
    max_impressions: int = 30_000,
    seed: int = 42,
    require_click: bool = True,
) -> pd.DataFrame:
    """Берём сэмпл impression'ов из Parquet, фильтруя coview строки.

    Возвращает DataFrame со строками (impression_key, source_article_id, clicked_target_ids[list[str]]).
    """
    rng = np.random.default_rng(seed)
    dataset = ds.dataset(parquet_path, format="parquet")

    cols = [
        "visit_id",
        "hit_dttm",
        "entity_type",
        "source_type",
        "source_article_id",
        "target_article_id",
        "target",
    ]
    cols = [c for c in cols if c in dataset.schema.names]
    need = {"visit_id", "source_article_id", "target_article_id", "target"}
    if not need.issubset(set(cols)):
        raise KeyError(f"Missing required cols in parquet: {need - set(cols)}")

    # читаем потоково и накапливаем показы, пока не наберём max_impressions
    out_rows: list[dict] = []
    seen_keys: set[str] = set()

    scanner = dataset.scanner(columns=cols, batch_size=250_000)
    for batch in scanner.to_batches():
        pdf = batch.to_pandas(strings_to_categorical=False)

        # убираем coview строки, чтобы не смешивать поведение и показы рекомендаций
        if "source_type" in pdf.columns:
            pdf = pdf[pdf["source_type"].astype(str) != "coview"]

        pdf["visit_id"] = pdf["visit_id"].map(safe_str)
        pdf["source_article_id"] = pdf["source_article_id"].map(safe_str)
        pdf["target_article_id"] = pdf["target_article_id"].map(safe_str)
        pdf["target"] = pd.to_numeric(pdf["target"], errors="coerce").fillna(0).astype(int)

        if "hit_dttm" in pdf.columns:
            pdf["hit_dttm"] = pd.to_datetime(pdf["hit_dttm"], errors="coerce", utc=True)
        if "entity_type" in pdf.columns:
            pdf["entity_type"] = pdf["entity_type"].map(safe_str)

        # impression key
        if "hit_dttm" in pdf.columns and "entity_type" in pdf.columns:
            pdf["impression_key"] = pdf["visit_id"] + "|" + pdf["source_article_id"] + "|" + pdf["hit_dttm"].astype(str) + "|" + pdf["entity_type"]
        elif "hit_dttm" in pdf.columns:
            pdf["impression_key"] = pdf["visit_id"] + "|" + pdf["source_article_id"] + "|" + pdf["hit_dttm"].astype(str)
        else:
            pdf["impression_key"] = pdf["visit_id"] + "|" + pdf["source_article_id"]

        # берём только кликающие показы (если требуется)
        if require_click:
            pdf = pdf[pdf["target"] == 1]

        if len(pdf) == 0:
            continue

        # агрегируем кликнутые target'ы внутри impression
        g = pdf.groupby(["impression_key", "source_article_id"], sort=False)["target_article_id"].apply(lambda s: list({safe_str(x) for x in s.tolist() if safe_str(x)}))
        for (imp_key, src_id), tgt_ids in g.items():
            if not src_id or not imp_key or not tgt_ids:
                continue
            if imp_key in seen_keys:
                continue
            seen_keys.add(imp_key)
            out_rows.append({"impression_key": imp_key, "source_article_id": src_id, "clicked_target_ids": tgt_ids})

        if len(out_rows) >= max_impressions:
            break

    # shuffle
    if len(out_rows) > 1:
        idx = np.arange(len(out_rows))
        rng.shuffle(idx)
        out_rows = [out_rows[i] for i in idx]

    return pd.DataFrame(out_rows[:max_impressions])


def recall_at_n_for_retrieval(
    art: Artifacts,
    imps: pd.DataFrame,
    *,
    N: int,
    coview_index: dict[str, list[tuple[str, float]]] | None,
) -> dict[str, float]:
    """Считает Recall@N для embeddings-only / coview-only / union."""
    hits_emb = 0
    hits_cov = 0
    hits_union = 0
    total = 0

    for _, row in imps.iterrows():
        src_id = safe_str(row["source_article_id"])
        tgt_ids = set(row["clicked_target_ids"])
        sidx = art.article_id_to_row.get(src_id)
        if sidx is None:
            continue

        # embeddings-only
        emb_idx, _ = get_similar_candidates(art, sidx, top_n=N, coview_index=None)
        emb_ids = {safe_str(art.df.iloc[i].get("article_id", "")) for i in emb_idx[:N]}

        # coview-only
        cov_ids: set[str] = set()
        if coview_index is not None and src_id in coview_index:
            cov_ids = {safe_str(t) for t, _ in coview_index[src_id][:N]}

        union_ids = (emb_ids | cov_ids)

        total += 1
        if tgt_ids & emb_ids:
            hits_emb += 1
        if tgt_ids & cov_ids:
            hits_cov += 1
        if tgt_ids & union_ids:
            hits_union += 1

    if total == 0:
        return {"emb": 0.0, "cov": 0.0, "union": 0.0}

    return {
        "emb": hits_emb / total,
        "cov": hits_cov / total,
        "union": hits_union / total,
    }


# --- Запуск (ручной):
# 1) Построить/загрузить coview index (если нужен)
# COVIEW_CACHE_PATH = 'coview_cache/coview_index.pkl.gz'
# coview_index = load_coview_index(COVIEW_CACHE_PATH) if os.path.exists(COVIEW_CACHE_PATH) else build_coview_index(LOGS_PARQUET_PATH)
#
# 2) Сэмпл кликающих показов
# imps = sample_impressions_from_logs(LOGS_PARQUET_PATH, max_impressions=20000)
#
# 3) Recall@N
# for N in [50, 100, 200, 400, 800]:
#     print(N, recall_at_n_for_retrieval(art, imps, N=N, coview_index=coview_index))


In [ ]:
# Путь к логам (Parquet). Можно оставить абсолютный.
LOGS_PARQUET_PATH = "/Users/nkozheko/Downloads/tj_session_w_target_full.parquet"

# Колонки, которые ожидаем в логах
_LTR_LOG_COLS = [
    "wuid",
    "visit_id",
    "hit_dttm",
    "source_article_id",
    "target_article_id",
    "entity_type",
    "article_position",
    "target",
    "target_read",
]


def _to_dt_utc(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", utc=True)


def load_ltr_logs_sample(
    parquet_path: str = LOGS_PARQUET_PATH,
    *,
    row_groups: int = 8,
    take_per_group: int = 250_000,
    seed: int = 42,
) -> pd.DataFrame:
    """Быстро берём репрезентативный сэмпл без чтения всего файла."""
    import numpy as _np
    import pyarrow.parquet as pq

    pf = pq.ParquetFile(parquet_path)
    rg_total = pf.metadata.num_row_groups
    rng = _np.random.default_rng(seed)
    rg_ids = sorted(set(rng.integers(0, rg_total, size=min(row_groups, rg_total)).tolist()))

    cols = [c for c in _LTR_LOG_COLS if c in pf.schema.names]
    need = {"visit_id","source_article_id","target_article_id","target"}
    if not need.issubset(set(cols)):
        raise KeyError(f"Missing required cols. Have: {cols}")

    chunks: list[pd.DataFrame] = []
    for i in rg_ids:
        t = pf.read_row_group(i, columns=cols)
        dfc = t.to_pandas(strings_to_categorical=False)
        if len(dfc) > take_per_group:
            dfc = dfc.sample(take_per_group, random_state=seed)
        chunks.append(dfc)

    out = pd.concat(chunks, ignore_index=True)
    out["visit_id"] = out["visit_id"].map(safe_str)
    out["source_article_id"] = out["source_article_id"].map(safe_str)
    out["target_article_id"] = out["target_article_id"].map(safe_str)
    if "wuid" in out.columns:
        out["wuid"] = out["wuid"].map(safe_str)
    if "entity_type" in out.columns:
        out["entity_type"] = out["entity_type"].map(safe_str)
    if "hit_dttm" in out.columns:
        out["hit_dttm"] = _to_dt_utc(out["hit_dttm"])

    out = out[(out["visit_id"] != "") & (out["source_article_id"] != "") & (out["target_article_id"] != "")]
    out["target"] = pd.to_numeric(out["target"], errors="coerce").fillna(0).astype(int)
    if "article_position" in out.columns:
        out["article_position"] = pd.to_numeric(out["article_position"], errors="coerce").fillna(-1).astype(int)

    return out


# === Подготовка фичей и обучение LTR (LightGBMRanker) ===

# Порядок фичей должен совпадать с тем, что используется в `rerank_and_diversify_two_stage`.
_LTR_FEATURE_NAMES = [
    # semantic relevance
    "sim",
    # candidate signals (log-scaled where needed)
    "cand_log_views",
    "cand_like_rate",
    "cand_comment_rate",
    "cand_log_comments",
    "cand_fav_rate",
    "cand_fresh",
    # pairwise (source-candidate)
    "same_author",
    "abs_age_diff_days",
    # match with source
    "same_dept",
    "same_rubric",
    # source context
    "src_log_views",
    "src_like_rate",
    "src_fresh",
    # position in shown list (for debiasing / learning)
    "pos",
]


def _safe_num(x: object) -> float:
    try:
        if pd.isna(x):
            return 0.0
        return float(x)
    except Exception:
        return 0.0


def _log1p_series(s: pd.Series) -> np.ndarray:
    v = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(np.float32).values
    v[v < 0] = 0
    return np.log1p(v).astype(np.float32)


def _rate_series(s: pd.Series) -> np.ndarray:
    return pd.to_numeric(s, errors="coerce").fillna(0.0).astype(np.float32).values


def prepare_ltr_feature_cache(art: Artifacts) -> dict[str, np.ndarray]:
    """Предвычисляет численные фичи по статьям для быстрых джойнов из логов."""
    d = art.df

    views = _log1p_series(d.get("article_stats__stats_views", 0.0))
    comments_log = _log1p_series(d.get("article_stats__stats_comments", 0.0))

    like_rate = _rate_series(d.get("article_stats__like_rate", 0.0))
    comment_rate = _rate_series(d.get("article_stats__comment_rate", 0.0))

    favs = pd.to_numeric(d.get("article_stats__stats_favorites", 0.0), errors="coerce").fillna(0.0).astype(np.float32).values
    views_raw = pd.to_numeric(d.get("article_stats__stats_views", 0.0), errors="coerce").fillna(0.0).astype(np.float32).values
    denom = np.maximum(views_raw, 1.0)
    fav_rate = (favs / denom).astype(np.float32)

    fresh = d.apply(freshness, axis=1).astype(np.float32).values

    age_days = pd.to_numeric(d.get("article_dates__days_since_published", 365.0), errors="coerce").fillna(365.0).astype(np.float32).values
    author_id = d.get("article_base__author_id", "").map(safe_str).astype(str).values

    return {
        "log_views": views,
        "log_comments": comments_log,
        "like_rate": like_rate,
        "comment_rate": comment_rate,
        "fav_rate": fav_rate,
        "fresh": fresh,
        "age_days": age_days,
        "author_id": author_id,
    }


def estimate_position_propensity(logs: pd.DataFrame, *, pos_col: str = "article_position", target_col: str = "target") -> dict[int, float]:
    """Оценивает propensity p(click|pos) по train логам.

    Используем простое сглаживание, чтобы не делить на ноль.
    """
    if pos_col not in logs.columns or target_col not in logs.columns:
        return {}

    tmp = logs[[pos_col, target_col]].copy()
    tmp[pos_col] = pd.to_numeric(tmp[pos_col], errors="coerce").fillna(-1).astype(int)
    tmp[target_col] = pd.to_numeric(tmp[target_col], errors="coerce").fillna(0).astype(float)

    g = tmp.groupby(pos_col, sort=False)[target_col]
    cnt = g.size().astype(float)
    sm = g.sum().astype(float)

    # Beta(1,1) smoothing
    prop = ((sm + 1.0) / (cnt + 2.0)).to_dict()
    return {int(k): float(v) for k, v in prop.items()}


def build_ltr_dataset_from_logs(
    art: Artifacts,
    logs: pd.DataFrame,
    *,
    max_impressions: int = 200_000,
    min_group_size: int = 2,
    keep_entity_types: set[str] | None = None,
    seed: int = 42,
    position_propensity: dict[int, float] | None = None,
    propensity_floor: float = 0.02,
) -> tuple[np.ndarray, np.ndarray, list[int], np.ndarray]:
    """Готовит (X, y, group) для обучения LTR.

    Query (= group) — один показ блока рекомендаций под статьёй-источником.
    Определяем его как (visit_id, source_article_id, hit_dttm, entity_type).

    Чтобы не грузить гигантский датасет в ноутбуке, берём подвыборку показов.
    """
    rng = np.random.default_rng(seed)

    need_cols = {"visit_id", "source_article_id", "target_article_id", "target"}
    if not need_cols.issubset(set(logs.columns)):
        raise KeyError(f"logs missing required cols: {need_cols - set(logs.columns)}")

    df = logs.copy()

    if keep_entity_types is not None and "entity_type" in df.columns:
        df = df[df["entity_type"].map(safe_str).isin(keep_entity_types)].copy()

    # impression key
    if "hit_dttm" in df.columns and "entity_type" in df.columns:
        df["impression_key"] = (
            df["visit_id"].map(safe_str)
            + "|"
            + df["source_article_id"].map(safe_str)
            + "|"
            + df["hit_dttm"].astype(str)
            + "|"
            + df["entity_type"].map(safe_str)
        )
    elif "hit_dttm" in df.columns:
        df["impression_key"] = df["visit_id"].map(safe_str) + "|" + df["source_article_id"].map(safe_str) + "|" + df["hit_dttm"].astype(str)
    else:
        # fallback (хуже): группируем по (visit, source)
        df["impression_key"] = df["visit_id"].map(safe_str) + "|" + df["source_article_id"].map(safe_str)

    # оставляем только пары, которые есть в aligned df/embeddings
    m_src = df["source_article_id"].map(lambda x: safe_str(x) in art.article_id_to_row)
    m_tgt = df["target_article_id"].map(lambda x: safe_str(x) in art.article_id_to_row)
    df = df[m_src & m_tgt].copy()

    # подвыборка показов
    keys = df["impression_key"].unique()
    if len(keys) > max_impressions:
        keys = rng.choice(keys, size=max_impressions, replace=False)
    df = df[df["impression_key"].isin(set(keys))].copy()

    # группировка
    g = df.groupby("impression_key", sort=False)

    X_parts: list[np.ndarray] = []
    y_parts: list[np.ndarray] = []
    w_parts: list[np.ndarray] = []
    group_sizes: list[int] = []

    prop = position_propensity or {}

    cache = prepare_ltr_feature_cache(art)

    for _, grp in g:
        if len(grp) < min_group_size:
            continue

        src_id = safe_str(grp["source_article_id"].iloc[0])
        sidx = art.article_id_to_row.get(src_id)
        if sidx is None:
            continue
        src_row = art.df.iloc[sidx]
        src_dept = safe_str(src_row.get("article_base__department", ""))
        src_rub = safe_str(src_row.get("article_base__rubric", ""))

        src_log_views = float(cache["log_views"][sidx])
        src_like_rate = float(cache["like_rate"][sidx])
        src_fresh = float(cache["fresh"][sidx])
        src_age_days = float(cache["age_days"][sidx])
        src_author = safe_str(cache["author_id"][sidx])

        # targets
        cand_ids = grp["target_article_id"].map(safe_str).tolist()
        crows = [art.article_id_to_row[cid] for cid in cand_ids]

        # sim
        vv = art.X[np.asarray(crows, dtype=np.int64)]
        sim = (vv @ art.X[sidx]).astype(np.float32)

        # candidate numeric features (из таблицы статей)
        cand_df = art.df.iloc[crows]

        cand_log_views = cache["log_views"][np.asarray(crows, dtype=np.int64)]
        cand_log_comments = cache["log_comments"][np.asarray(crows, dtype=np.int64)]
        cand_like_rate = cache["like_rate"][np.asarray(crows, dtype=np.int64)]
        cand_comment_rate = cache["comment_rate"][np.asarray(crows, dtype=np.int64)]
        cand_fav_rate = cache["fav_rate"][np.asarray(crows, dtype=np.int64)]
        cand_fresh = cache["fresh"][np.asarray(crows, dtype=np.int64)]
        cand_age_days = cache["age_days"][np.asarray(crows, dtype=np.int64)]
        cand_author = np.asarray(cache["author_id"], dtype=object)[np.asarray(crows, dtype=np.int64)]

        same_author = (pd.Series(cand_author).map(safe_str).values == src_author).astype(np.float32)
        abs_age_diff_days = np.abs(cand_age_days.astype(np.float32) - np.float32(src_age_days)).astype(np.float32)

        same_dept = (cand_df.get("article_base__department", "").map(safe_str).values == src_dept).astype(np.float32)
        same_rub = (cand_df.get("article_base__rubric", "").map(safe_str).values == src_rub).astype(np.float32)

        if "article_position" in grp.columns:
            pos = pd.to_numeric(grp["article_position"], errors="coerce").fillna(-1).astype(np.float32).values
        else:
            pos = np.zeros(len(grp), dtype=np.float32)

        # propensity weighting by position (debias)
        if prop:
            pos_int = pos.astype(np.int32)
            p = np.asarray([prop.get(int(pp), propensity_floor) for pp in pos_int], dtype=np.float32)
            p = np.clip(p, propensity_floor, 1.0)
            w = (1.0 / p).astype(np.float32)
        else:
            w = np.ones(len(grp), dtype=np.float32)

        X = np.column_stack([
            sim,
            cand_log_views,
            cand_like_rate,
            cand_comment_rate,
            cand_log_comments,
            cand_fav_rate,
            cand_fresh,
            same_author,
            abs_age_diff_days,
            same_dept,
            same_rub,
            np.full(len(grp), float(src_log_views), dtype=np.float32),
            np.full(len(grp), float(src_like_rate), dtype=np.float32),
            np.full(len(grp), float(src_fresh), dtype=np.float32),
            pos,
        ]).astype(np.float32)

        y = pd.to_numeric(grp["target"], errors="coerce").fillna(0).astype(np.float32).values

        X_parts.append(X)
        y_parts.append(y)
        w_parts.append(w)
        group_sizes.append(len(grp))

    if not group_sizes:
        raise ValueError("No groups prepared. Check that ids align and impression_key is correct.")

    X_all = np.vstack(X_parts)
    y_all = np.concatenate(y_parts)
    w_all = np.concatenate(w_parts) if w_parts else np.ones(len(y_all), dtype=np.float32)
    return X_all, y_all, group_sizes, w_all


def train_ltr_ranker_lgbm(
    X: np.ndarray,
    y: np.ndarray,
    group: list[int],
    *,
    sample_weight: np.ndarray | None = None,
    seed: int = 42,
):
    """Обучает LightGBM LambdaRank. Возвращает объект с `.predict()`."""
    import lightgbm as lgb

    model = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=350,
        learning_rate=0.06,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(X, y, group=group, sample_weight=sample_weight)
    return model


def train_ltr_ranker_catboost(
    X: np.ndarray,
    y: np.ndarray,
    group: list[int],
    *,
    sample_weight: np.ndarray | None = None,
    seed: int = 42,
):
    """Обучает CatBoostRanker. Возвращает объект с `.predict()`.

    Важно: здесь используем численные фичи (без категорий). Если добавите
    категориальные признаки — CatBoost станет ещё более уместным.
    """
    from catboost import CatBoostRanker, Pool

    # group_id нужен на уровне строк; восстановим из group sizes
    group_id = np.repeat(np.arange(len(group), dtype=np.int64), np.asarray(group, dtype=np.int64))

    pool = Pool(
        data=X,
        label=y,
        group_id=group_id,
        weight=sample_weight,
    )

    model = CatBoostRanker(
        loss_function="YetiRank",
        iterations=800,
        learning_rate=0.05,
        depth=8,
        random_seed=seed,
        verbose=200,
    )
    model.fit(pool)
    return model


def train_ltr_ranker(
    backend: str,
    X: np.ndarray,
    y: np.ndarray,
    group: list[int],
    *,
    sample_weight: np.ndarray | None = None,
    seed: int = 42,
):
    backend = safe_str(backend).lower()
    if backend in {"lgbm", "lightgbm"}:
        return train_ltr_ranker_lgbm(X, y, group, sample_weight=sample_weight, seed=seed)
    if backend in {"cat", "catboost"}:
        return train_ltr_ranker_catboost(X, y, group, sample_weight=sample_weight, seed=seed)
    raise ValueError(f"Unknown backend: {backend}")


# Пример использования (запускайте вручную):
# MODEL_BACKEND = 'lgbm'  # или 'catboost'
# logs = load_ltr_logs_sample(LOGS_PARQUET_PATH, row_groups=12, take_per_group=200_000)
# X_ltr, y_ltr, group_ltr = build_ltr_dataset_from_logs(art, logs, max_impressions=120_000)
# ltr = train_ltr_ranker(MODEL_BACKEND, X_ltr, y_ltr, group_ltr)


In [ ]:

# (пусто) дубль ячейки выше — оставлено намеренно пустым


### Time-split валидация LTR (nDCG@6, MRR@6)

Схема:

- берём сэмпл логов показов (не `coview` строки)
- делаем сплит по времени (train до `SPLIT_DATE_UTC`, test после)
- строим LTR датасет (query = impression)
- обучаем LightGBM и CatBoost
- считаем nDCG@6 и MRR@6 на test


### Быстрая проверка на одной статье (sanity check)

Проверяет, что сборка карусели даёт **K=12** с **3 explore** (2 same‑dept другая рубрика + 1 cross‑dept с блэклистом).


In [ ]:
# Пример: подставьте конкретный article_id при желании
source_article_id = "19f22d82-87ee-4aa3-820e-6d4f60ca76a7"  # можно заменить

# 1) Загрузите эмбеддинги (ожидается, что вы уже сделали build_embeddings_memmap)
X_ids, X = load_embeddings_memmap('embeddings_cache')

# 2) Fit retrieval
# Добавим title_norm (если ещё не добавляли)
df_local = df.copy()
if "title_norm" not in df_local.columns and "article_base__title" in df_local.columns:
    df_local["title_norm"] = df_local["article_base__title"].map(normalize_title_for_dedup)

art = fit_retriever_from_text_embeddings(df_local, X_ids, X)

# 3) Explore pools
per_dept, global_pool = build_explore_pools(art.df, top_m=3000)

# 3b) (Опционально) coview индекс для retrieval similar
USE_COVIEW_RETRIEVAL = False
coview_index = None
COVIEW_CACHE_PATH = "coview_cache/coview_index.pkl.gz"

if USE_COVIEW_RETRIEVAL:
    if os.path.exists(COVIEW_CACHE_PATH):
        coview_index = load_coview_index(COVIEW_CACHE_PATH)
    else:
        coview_index = build_coview_index(
            LOGS_PARQUET_PATH,
            top_k_per_source=200,
            min_count=2,
            source_type_value="coview",
            max_rows=3_000_000,  # ограничение для ноутбука; снимите для полного прогона
            score_mode="p_b_given_a",
            alpha=1.0,
        )
        save_coview_index(coview_index, COVIEW_CACHE_PATH)

# 4) Найдём source_idx в aligned df
sid = safe_str(source_article_id)
if sid not in art.article_id_to_row:
    raise KeyError(f"source_article_id not found in aligned df/embeddings: {sid}")
source_idx = art.article_id_to_row[sid]

# 5) Два пула кандидатов
sim_idx, sim_sims = get_similar_candidates(
    art,
    source_idx,
    top_n=800,
    coview_index=coview_index,
    coview_top_m=200,
)
explore_df = get_explore_candidates(art, source_idx, per_dept_pool=per_dept, global_pool=global_pool)

# 6) (Опционально) обучаем LTR и применяем в финальном rerank
USE_LTR = False
MODEL_BACKEND = "lgbm"  # или "catboost"
ltr_model = None

if USE_LTR:
    logs = load_ltr_logs_sample(LOGS_PARQUET_PATH, row_groups=10, take_per_group=200_000)
    pos_prop = estimate_position_propensity(logs)
    X_ltr, y_ltr, group_ltr, w_ltr = build_ltr_dataset_from_logs(
        art,
        logs,
        max_impressions=120_000,
        position_propensity=pos_prop,
        propensity_floor=0.02,
    )
    ltr_model = train_ltr_ranker(MODEL_BACKEND, X_ltr, y_ltr, group_ltr, sample_weight=w_ltr)

# 7) Сборка карусели
picked = rerank_and_diversify_two_stage(
    art,
    source_idx,
    sim_idx,
    sim_sims,
    explore_df,
    k=12,
    explore_slots=3,
    ltr_model=ltr_model,
)

# 7) Отчёт
src_row = art.df.iloc[source_idx]
src_dept = safe_str(src_row.get('article_base__department',''))
src_rub = safe_str(src_row.get('article_base__rubric',''))

picked_df = pd.DataFrame(picked)
print('picked:', len(picked_df))
print(picked_df['mix'].value_counts(dropna=False).to_string())

# Проверки правил
exp = picked_df[picked_df['mix']=='explore'].copy()
exp_same = exp[exp['candidate_department']==src_dept]
exp_cross = exp[exp['candidate_department']!=src_dept]

# 2 same-dept другой рубрики
viol_same_rub = 0
if src_rub:
    viol_same_rub = int((exp_same['candidate_rubric'] == src_rub).sum())

# cross-dept не из блэклиста
forbidden = DEPT_CROSS_BLACKLIST.get(src_dept, set())
viol_cross = int(exp_cross['candidate_department'].isin(forbidden).sum())

print('same_dept_explore:', len(exp_same), 'cross_dept_explore:', len(exp_cross))
print('violations: same_rubric=', viol_same_rub, 'cross_blacklist=', viol_cross)

display(pd.DataFrame([{
    'source_article_id': sid,
    'source_title': safe_str(src_row.get('article_base__title','')),
    'source_department': src_dept,
    'source_rubric': src_rub,
}]))

display(picked_df[['mix','candidate_title','candidate_department','candidate_rubric','score','similarity']].head(20))


### Оффлайн‑оценка retrieval: попадает ли клик в candidate pool (Recall@N)

Идея: по логам показов берём клики (target=1) и проверяем, попал ли кликнутый `target_article_id` в пул кандидатов для `source_article_id`.

Сравниваем 3 режима retrieval:

- **embeddings‑only**
- **coview‑only**
- **union** (embeddings + coview)


### Экспорт sample: `recs_i2i_sample.csv`


In [ ]:
# test cell


In [ ]:
def build_i2i_sample_csv(
    *,
    out_path: str = "recs_i2i_sample.csv",
    max_articles: int = 2000,
    k: int = 20,
    explore_slots: int = 4,
    topn_similar: int = 800,
    ltr_model: object | None = None,
    coview_index: dict[str, list[tuple[str, float]]] | None = None,
    coview_top_m: int = 200,
    max_candidate_age_days: float | None = None,
) -> pd.DataFrame:
    df_local = df.head(max_articles).copy()
    if "article_base__title" in df_local.columns:
        df_local["title_norm"] = df_local["article_base__title"].map(normalize_title_for_dedup)
    else:
        df_local["title_norm"] = ""

    X_ids, X = load_embeddings_memmap('embeddings_cache')
    art = fit_retriever_from_text_embeddings(df_local, X_ids, X)

    per_dept, global_pool = build_explore_pools(art.df, top_m=3000)

    rows: list[dict] = []
    for src_idx in range(art.df.shape[0]):
        sim_idx, sim_sims = get_similar_candidates(
            art,
            src_idx,
            top_n=topn_similar,
            coview_index=coview_index,
            coview_top_m=200,
        )
        explore_df = get_explore_candidates(art, src_idx, per_dept_pool=per_dept, global_pool=global_pool)

        picked = rerank_and_diversify_two_stage(
            art,
            src_idx,
            sim_idx,
            sim_sims,
            explore_df,
            k=k,
            explore_slots=explore_slots,
            ltr_model=ltr_model,
            max_candidate_age_days=max_candidate_age_days,
        )
        for r, row in enumerate(picked, start=1):
            row['rank'] = r
            rows.append(row)

    out = pd.DataFrame(rows)
    cols = [
        'source_article_id','candidate_article_id','rank','mix','score','similarity',
        'candidate_title','candidate_department','candidate_rubric'
    ]
    out = out[cols].sort_values(['source_article_id','rank'], ascending=[True, True])
    out.to_csv(out_path, index=False)
    return out


# Пример вызова (не запускаем автоматически):
# build_embeddings_memmap('user_articles_embeddings.csv', out_dir='embeddings_cache')
#
# # coview retrieval (опционально): построить один раз и закэшировать
# COVIEW_CACHE_PATH = 'coview_cache/coview_index.pkl.gz'
# if os.path.exists(COVIEW_CACHE_PATH):
#     coview_index = load_coview_index(COVIEW_CACHE_PATH)
# else:
#     coview_index = build_coview_index(
#         LOGS_PARQUET_PATH,
#         top_k_per_source=200,
#         min_count=2,
#         source_type_value='coview',
#         score_mode='p_b_given_a',
#         alpha=1.0,
#     )
#     save_coview_index(coview_index, COVIEW_CACHE_PATH)
#
# recs_df = build_i2i_sample_csv(
#     out_path='recs_i2i_sample.csv',
#     max_articles=2000,
#     k=12,
#     explore_slots=3,
#     coview_index=coview_index,
#     coview_top_m=200,
# )


### Просмотр примеров


### Оффлайн‑оценка retrieval: попадает ли клик в candidate pool (Recall@N)

Идея: по логам показов берём клики (target=1) и проверяем, попал ли кликнутый `target_article_id` в пул кандидатов для `source_article_id`.

Сравниваем 3 режима retrieval:

- **embeddings‑only**
- **coview‑only**
- **union** (embeddings + coview)


In [ ]:
def show_carousel_example(
    source_article_id: str,
    *,
    recs_df: pd.DataFrame | None = None,
    recs_csv_path: str = 'recs_i2i_sample.csv',
    top_k: int = 12,
) -> None:
    sid = safe_str(source_article_id)
    src = df[df['article_id'] == sid]
    if len(src) == 0:
        raise KeyError(f'Unknown article_id: {sid}')
    src_row = src.iloc[0]

    if recs_df is None:
        recs_df = pd.read_csv(recs_csv_path)

    recs = recs_df[recs_df['source_article_id'] == sid].copy().sort_values('rank').head(top_k)

    display(pd.DataFrame([{
        'article_id': sid,
        'title': safe_str(src_row.get('article_base__title','')),
        'department': safe_str(src_row.get('article_base__department','')),
        'rubric': safe_str(src_row.get('article_base__rubric','')),
    }]))

    display(recs[[
        'rank','mix','candidate_article_id','candidate_title','candidate_department','candidate_rubric','score','similarity'
    ]])


# Пример использования (не запускаем автоматически):
# recs_df = pd.read_csv('recs_i2i_sample.csv')
# show_carousel_example(recs_df['source_article_id'].dropna().sample(1).iloc[0], recs_df=recs_df, top_k=12)


### Сравнение двух алгоритмов на одной статье

Сравниваем:

- **Наша система**: retrieval (embeddings + опционально coview) + explore + правила + (опционально) LTR.
- **Baseline (как в проде)**: **только семантическая близость** (top‑K по cosine в эмбеддингах), без explore и без правил.


In [ ]:
def baseline_semantic_only(
    art: Artifacts,
    source_idx: int,
    *,
    k: int = 20,
    max_candidate_age_days: float | None = None,
) -> list[dict]:
    """Baseline: top-K по cosine similarity в эмбеддингах (без explore/правил)."""
    cap = int(art.df.shape[0])
    if max_candidate_age_days is None:
        n_neighbors = min(k + 1, cap)
    else:
        n_neighbors = min(max(k * 40, k + 5), cap)
    distances, indices = art.nn.kneighbors(
        art.X[source_idx : source_idx + 1],
        n_neighbors=n_neighbors,
    )
    distances = distances.ravel()
    indices = indices.ravel()

    out: list[dict] = []
    src = art.df.iloc[source_idx]
    src_id = safe_str(src.get("article_id", ""))

    for dist, idx in zip(distances, indices):
        if int(idx) == int(source_idx):
            continue
        cid = safe_str(art.df.iloc[int(idx)].get("article_id", ""))
        if not cid:
            continue
        sim = float(1.0 - float(dist))
        r = art.df.iloc[int(idx)]
        if max_candidate_age_days is not None:
            _ac = "article_dates__days_since_published"
            if _ac in art.df.columns:
                _days = r.get(_ac, np.nan)
                if pd.notna(_days) and float(_days) > float(max_candidate_age_days):
                    continue
        out.append({
            "source_article_id": src_id,
            "candidate_article_id": cid,
            "rank": len(out) + 1,
            "mix": "semantic_only",
            "score": sim,
            "similarity": sim,
            "candidate_title": safe_str(r.get("article_base__title", "")),
            "candidate_department": safe_str(r.get("article_base__department", "")),
            "candidate_rubric": safe_str(r.get("article_base__rubric", "")),
        })
        if len(out) >= k:
            break

    return out


def compare_two_systems_one_article(
    source_article_id: str,
    *,
    k: int = 12,
    explore_slots: int = 3,
    topn_similar: int = 800,
    use_coview: bool = False,
    use_ltr: bool = False,
    coview_max_rows: int | None = 3_000_000,
    ltr_backend: str = "lgbm",
    max_candidate_age_days: float | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Возвращает (our_df, baseline_df) для одной статьи."""
    # align / artifacts
    df_local = df.copy()
    if "title_norm" not in df_local.columns and "article_base__title" in df_local.columns:
        df_local["title_norm"] = df_local["article_base__title"].map(normalize_title_for_dedup)

    X_ids, X = load_embeddings_memmap('embeddings_cache')
    art = fit_retriever_from_text_embeddings(df_local, X_ids, X)

    sid = safe_str(source_article_id)
    if sid not in art.article_id_to_row:
        raise KeyError(f"source_article_id not found in aligned df/embeddings: {sid}")
    source_idx = art.article_id_to_row[sid]

    # explore pools
    per_dept, global_pool = build_explore_pools(art.df, top_m=3000)

    # coview index (optional)
    coview_index = None
    if use_coview:
        cache_path = "coview_cache/coview_index.pkl.gz"
        if os.path.exists(cache_path):
            coview_index = load_coview_index(cache_path)
        else:
            coview_index = build_coview_index(
                LOGS_PARQUET_PATH,
                top_k_per_source=200,
                min_count=2,
                source_type_value="coview",
                max_rows=coview_max_rows,
                score_mode="p_b_given_a",
                alpha=1.0,
            )
            save_coview_index(coview_index, cache_path)

    # LTR model (optional)
    ltr_model = None
    if use_ltr:
        logs = load_ltr_logs_sample(LOGS_PARQUET_PATH, row_groups=10, take_per_group=200_000)
        pos_prop = estimate_position_propensity(logs)
        X_ltr, y_ltr, group_ltr, w_ltr = build_ltr_dataset_from_logs(
            art,
            logs,
            max_impressions=120_000,
            position_propensity=pos_prop,
            propensity_floor=0.02,
        )
        ltr_model = train_ltr_ranker(ltr_backend, X_ltr, y_ltr, group_ltr, sample_weight=w_ltr)

    # our system
    sim_idx, sim_sims = get_similar_candidates(
        art,
        source_idx,
        top_n=topn_similar,
        coview_index=coview_index,
        coview_top_m=200,
    )
    explore_df = get_explore_candidates(art, source_idx, per_dept_pool=per_dept, global_pool=global_pool)

    our = rerank_and_diversify_two_stage(
        art,
        source_idx,
        sim_idx,
        sim_sims,
        explore_df,
        k=k,
        explore_slots=explore_slots,
        ltr_model=ltr_model,
        max_candidate_age_days=max_candidate_age_days,
    )
    our_df = pd.DataFrame(our)
    our_df.insert(2, "rank", np.arange(1, len(our_df) + 1))

    # baseline
    base = baseline_semantic_only(
        art,
        source_idx,
        k=k,
        max_candidate_age_days=max_candidate_age_days,
    )
    base_df = pd.DataFrame(base)

    # quick overlap stats
    a = set(our_df["candidate_article_id"].tolist())
    b = set(base_df["candidate_article_id"].tolist())
    inter = len(a & b)
    print(f"Overlap: {inter}/{k} (Jaccard={inter/max(len(a|b),1):.3f})")

    cols = ["rank", "mix", "candidate_title", "candidate_department", "candidate_rubric", "score", "similarity"]
    display(pd.DataFrame([{
        "source_article_id": sid,
        "source_title": safe_str(art.df.iloc[source_idx].get('article_base__title','')),
        "source_department": safe_str(art.df.iloc[source_idx].get('article_base__department','')),
        "source_rubric": safe_str(art.df.iloc[source_idx].get('article_base__rubric','')),
    }]))

    print("\nOUR SYSTEM")
    display(our_df[cols].head(k))

    print("\nBASELINE: semantic-only")
    display(base_df[cols].head(k))

    return our_df, base_df


# Пример запуска (замени id на нужный)
# compare_two_systems_one_article(
#     "19f22d82-87ee-4aa3-820e-6d4f60ca76a7",
#     k=12,
#     explore_slots=3,
#     use_coview=True,
#     use_ltr=False,
# )
